In [ ]:
# Rankine-Hugoniot Jump Conditions and Shock Cut (paper figures)
#
# Reads the data cached by notebooks/rh_conds/rh_conds_mime.ipynb and reproduces the
# two figures used in the previous paper's rh_conds.ipynb - "rh" and "shock_cut" - here
# named "rh_mime" and "shock_cut_mime". This notebook has no graphet/TristanRun
# dependency - it only needs the saved NetCDF file.

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

# HARD CODED WARNING
comp_to_di = 1 / np.sqrt(50)


def save_polished_fig(name, dpi=300, metadata=None):
    plt.savefig(f"/scratch/gpfs/ar3676/plots/{name}", bbox_inches='tight', dpi=dpi, metadata=metadata)


findex = 253 #250  # must match the findex used when the data was saved


data = xr.open_dataset(f"../rh_conds/rh_conds_data_{findex}.nc")

X_CUTOFF = data.attrs['X_CUTOFF']
xlim = data.attrs['xlim']
x_position_of_min_pb = data.attrs['x_position_of_min_pb']
boundary = data.attrs['boundary']

print(f"Loaded RH data for findex={findex}: theta = {data.attrs['angle_in_deg']:.2f} deg "
      f"({data.attrs['theta_method']}), steps = {list(data.attrs['my_steps'])}")

In [ ]:
# Six-Panel RH Plot ("rh_mime")

xmax = 85 / comp_to_di #XCUTOFF * xlim

def plot_squant(axis, quantity):
    right_val = data[quantity].sel(x=xmax, method="nearest")
    axis.plot(data.x * comp_to_di, data[quantity] / right_val)


eqn_fsize = 18
AXIS_FONTSIZE = plt.rcParams['font.size'] * 0.7  # tick/axis-label size, 30% smaller than default

fig, axes = plt.subplots(3, 2, figsize=(8, 4), sharex=True)
ax1, ax2, ax3, ax4, ax5, ax6 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]
plt.subplots_adjust(hspace=0, wspace=0.2)

# Mass

plot_squant(ax1, 'mass_flux')
ax1.text(0.45, 0.10, r"$\rho v_n$", transform=ax1.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax1.set_ylim(0, 2.2)

# Pressure

plot_squant(ax2, 'pressure')
ax2.set_ylim(0, 1.4)
ax2.text(0.35, 0.10, r"$\rho v_n^2 + P_g + P_B$", transform=ax2.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Energy

plot_squant(ax3, 'energy')
energy_expression = r'$\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}$'
ax3.text(0.15, 0.10, energy_expression, transform=ax3.transAxes, fontsize=12, verticalalignment='bottom')
ax3.set_ylim(0, 2.2)

# B_n

plot_squant(ax4, 'B_n')
ax4.text(0.45, 0.10, '$B_n$', transform=ax4.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax4.set_ylim(0, 2.2)

# Poynting Flux

plot_squant(ax5, 'poynting')
ax5.text(0.35, 0.10, r'$\rho v_n v_t - \frac{B_n B_t} {4 \pi}$', transform=ax5.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax5.set_ylim(0, 2.2)

# Electric Field

plot_squant(ax6, 'E_ideal')
ax6.text(0.35, 0.10, '$B_n v_t - v_n B_t$', transform=ax6.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax6.set_ylim(0, 2.2)

for ax_row in axes:
    for ax in ax_row:
        ax.set_xlim(x_position_of_min_pb * comp_to_di, xmax * comp_to_di)
        ax.axvline(boundary * comp_to_di, color='green', linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=AXIS_FONTSIZE)

for ax in [ax3, ax6]:
    ax.set_xlabel("$x$ [$d_i$]", fontsize=AXIS_FONTSIZE)

# Panel labels (a)-(f), reading left-to-right then top-to-bottom
panel_label_fsize = plt.rcParams['font.size'] * 0.9
panel_order = [ax1, ax4, ax2, ax5, ax3, ax6]
panel_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']
for ax, lab in zip(panel_order, panel_labels):
    ax.text(0.02, 0.92, lab, transform=ax.transAxes, fontsize=panel_label_fsize,
            va='top', ha='left')

# NOTE: save before show() - under the inline backend, plt.show() closes the
# figure, which would make a post-show savefig() silently save a blank canvas.
save_polished_fig("rh_mime")

plt.show()


In [ ]:
from scipy.ndimage import gaussian_filter1d

# Shock Cut ("shock_cut_mime")

#max_jz = data.jz.max()


fig, ax = plt.subplots(figsize=(6, 4))

ax.axvline(boundary * comp_to_di, color='green', linestyle='--')
ax.plot(data.x * comp_to_di, data.ndens, label="$n / n_0$")
ax.plot(data.x * comp_to_di, data.pn, label="$P_g / P_{B0}$")
ax.plot(data.x * comp_to_di, data.pb, label="$P_B / P_{B0}$")
#ax.plot(data.x * comp_to_di, data.jz / max_jz, label=r"$j_z / j_{max}$")

dx_di = float(np.diff(data.x).mean()) * comp_to_di
jz_smoothed = gaussian_filter1d(
    data.jz.values,
    sigma=0.5 / dx_di,
    mode="nearest",
)

x_plot = data.x.values * comp_to_di
plot_mask = (
    (x_plot >= x_position_of_min_pb * comp_to_di)
    & (x_plot <= xmax * comp_to_di)
)

max_jz = jz_smoothed[plot_mask].max()

ax.plot(
    data.x.values * comp_to_di,
    jz_smoothed / max_jz,
    label=r"$j_z / j_{max}$",
)

#ax.axvspan(
#    x_position_of_min_pb * comp_to_di,
#    xmax * comp_to_di,
#    color="gray",
#    alpha=0.25,
#    label="Range above",
#)

ax.set_ylim(-0.25, 2.5)
ax.set_xlim(x_position_of_min_pb * comp_to_di, xmax * comp_to_di) #xlim * comp_to_di * X_CUTOFF)
ax.set_xlabel("$x$ [$d_i$]", fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend()

ax.text(0.015, 0.97, '(g)', transform=ax.transAxes, fontsize=14, va='top', ha='left')

plt.tight_layout()

# NOTE: save before show() - under the inline backend, plt.show() closes the
# figure, which would make a post-show savefig() silently save a blank canvas.
save_polished_fig("shock_cut_mime")

plt.show()


In [ ]:
# Combined Figure ("combined_cut") - rh_mime stacked on top of shock_cut_mime
#
# Pastes the two already-rendered PNGs together vertically, centering each
# horizontally on the combined canvas (they don't need to share an x-axis).

from PIL import Image

rh_img = Image.open("/scratch/gpfs/ar3676/plots/rh_mime.png")
shock_img = Image.open("/scratch/gpfs/ar3676/plots/shock_cut_mime.png")

combined_width = max(rh_img.width, shock_img.width)
combined_height = rh_img.height + shock_img.height

combined = Image.new("RGBA", (combined_width, combined_height), (255, 255, 255, 255))
combined.paste(rh_img, ((combined_width - rh_img.width) // 2, 0), rh_img)
combined.paste(shock_img, ((combined_width - shock_img.width) // 2, rh_img.height), shock_img)

combined.save("/scratch/gpfs/ar3676/plots/combined_cut.png")

combined
